## *Week 7: Information Extraction and NER*

|*Name:*         |	Rubab Qaiser                                       |
|----------------|-----------------------------------------------------|
|*Course:*       |	Introduction to the Applied Artificial Intelligence|
|*Semester:*     |	BS 8th Semester                                    |
|*Week: *        |	Week 7                                             |
|*Project:*      |	Information Extraction + Named Entity Recognition  |
|*Lab Duration:* |	90 minutes                                         |

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/dataset_metadata.json
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00157.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00740.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00061.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00056.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00224.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00217.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00672.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00171.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00982.box
/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/boxes/PKR-00954.box
/kaggle/input/datasets/rubabq6

### *Lab Overview*

*Goal:* Transform unstructured text into structured data. Extract specific information(dates,accounts,names,organizations) from documents using Regular Expression and Named Entity Recognition.Build a complete extraction pipeline combining OCR, regex,and NER.

## *spaCy:*
spaCy is a powerful open-source Python library for Natural Language Processing (NLP).It helps computers read text the way humans do.

*spaCy* runs a pipeline of components such as:
---
- Tokenizer:Break text into words and punctuation.
- Tagger:Use to tag word, such as parts of speech tagger.e.g Apple → Proper Noun
- Parser:Understand sentance structure.it determine the relationship between s,o and v
- NER:Detect important entities.e.g Apple → Organization,London → Location
-Lemmatizer:Convert words to their base form such as running → run, good->better
---

*Common Applications:*
1.Resume parsing
2.Sentiment analysis
3.Document classification
4.Keyword extraction

*Popular spaCy Models:*
- Popular spaCy Models
- en_core_web_sm → small, fast
- en_core_web_md → medium, includes word vectors
- en_core_web_lg → large, more accurate
- en_core_web_trf → transformer-based, highest accuracy

In [2]:
pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [1]:
!python -m spacy validate

✔ Loaded compatibility table

================ Installed pipeline packages (spaCy v3.8.11) ================
ℹ spaCy installation: /usr/local/lib/python3.12/dist-packages/spacy

NAME             SPACY            VERSION                            
en_core_web_sm   >=3.8.0,<3.9.0   3.8.0   ✔



In [3]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 52.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
import spacy
nlp=spacy.load('en_core_web_md') #en_core_md is a small English model(13B)
print('spacy ready!')

spacy ready!


### *PART 1:REGULAR EXPRESSION*

### *Task 1.1:Extract Dates:*

In [13]:
import re

def extract_dates(text):
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',                      # MM/DD/YYYY or DD/MM/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',                      # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'                           # YYYY-MM-DD
    ]

    dates = []

    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)

    return dates

text = 'Invoice date: 03/15/2024. Due: March 30, 2024'
print(extract_dates(text))

['03/15/2024', 'March 30, 2024']


### *Task 1.2:Extract Currency Amounts*

In [14]:
import re

def extract_amounts(text):
    pattern = r'\$?\d+(?:,\d{3})*(?:\.\d{2})?'
    
    amounts = re.findall(pattern, text)
    
    cleaned = []
    for amount in amounts:
        clean = amount.replace('$', '').replace(',', '')
        cleaned.append(float(clean))
    
    return cleaned

# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 112.45"
print(extract_amounts(text))

[1250.5, 125.05, 112.45]


### *Task 1.3:Extract Invoice/Order Numbers*

In [16]:
import re

def extract_invoice(text):
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if match.groups():
                return match.group(1)
            else:
                return match.group(0)

    return None

text = "Invoice Number: INV-2024-001"
print(extract_invoice(text))

INV-2024-001


## *PART 2:NAMED IDENTITY RECOGNITION*

### *Task 2.1:Basic NER with spaCy*

In [17]:
import spacy
#load model
nlp=spacy.load('en_core_web_md')
#sample invoice 
text="""Invoice from Acme Corporation 123 Main 123 Main street,New York, NY 10001
contact: John Smith (john@acme.com) Date:March 15,2004, Amount Due: $1,250.50"""
doc=nlp(text)
print('Found Entities:')
for ent in doc.ents:
    print(f'{ent.text:20} {ent.label_:15} {spacy.explain(ent.label_)}')

Found Entities:
123                  CARDINAL        Numerals that do not fall under another type
New York             GPE             Countries, cities, states
NY 10001             ORG             Companies, agencies, institutions, etc.
John Smith           PERSON          People, including fictional
March 15,2004        DATE            Absolute or relative dates or periods
1,250.50             MONEY           Monetary values, including unit


### *Task 2.2: Extract Specific Entity Types*

In [25]:
def extract_entities(text):
    doc=nlp(text)
    entities={
        'persons':[],
        'organizations':[],
        'locations':[],
        'dates':[],
        'money':[]
    }
    for ent in doc.ents:
        if ent.label_ =='PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE','LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ =='DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)
        
    return entities
    
results=extract_entities(text)
for entity_type, values in results.items():
    print(f'{entity_type}: {values}')


persons: ['John Smith']
organizations: ['NY 10001']
locations: ['New York']
dates: ['March 15,2004']
money: ['1,250.50']


### *Task 2.3: Visualize Entities with displaCy

In [21]:
from spacy import displacy

displacy.render(doc, style='ent', jupyter=True)

# Generate HTML for saving
html = displacy.render(doc, style='ent', page=True)

with open('entities.html', 'w', encoding='utf-8') as f:
    f.write(str(html))

print("Visualization saved to entities.html")

Visualization saved to entities.html


## *PART 3: COMPLETE EXTRACTION PIPELINE*

### *Task 3.1:Build Invoice Processor:*

In [26]:
import pytesseract
from PIL import Image
import json
def process_invoice(image_path):
    #OCR->Extraction->JSON
    img=Image.open(image_path)
    text=pytesseract.image_to_string(img)
    #extract with regex
    invoice_data={
        'invoice_number':extract_invoice(text),
        'dates':extract_dates(text),
        'amounts':extract_amounts(text)
    }
    #extract with NER
    entities=extract_entities(text)
    invoice_data.update(entities)
    #post-process
    if invoice_data['amounts']:
        invoice_data['total_amount']=max(invoice_data['amounts'])
        if invoice_data['dates']:
            invoice_data['invoice_date']=invoice_data['dates'][0]
    return invoice_data

result=process_invoice('/kaggle/input/datasets/rubabq66/receiptdataset/receipt_dataset_1000/images/PKR-00001.png')
print(json.dumps(result,indent=2))
    
    

{
  "invoice_number": null,
  "dates": [],
  "amounts": [
    2026.0,
    4.0,
    2.0,
    1.0,
    9.0,
    18.0,
    8.0,
    11341.97,
    12.0,
    0.0,
    1361.04,
    12703.01,
    5219.33,
    6122.64
  ],
  "persons": [
    "Kurta Men\n\n \n\nSubtotal"
  ],
  "organizations": [
    "KHAADI",
    "Dolmen Mall"
  ],
  "locations": [
    "Khaadi"
  ],
  "money": [],
  "total_amount": 12703.01
}


### *Task 3.2:Save Results as JSON*

In [27]:
output_file='extracted_data.json'
with open(output_file,'w') as f:
    json.dump(result,f,indent=2)

print(f'Results saved to {output_file}')

Results saved to extracted_data.json


# NLP Document Processing through RegEx and NER:

## Overview

This notebook demonstrates the development of an intelligent document-processing pipeline using Python and Natural Language Processing (NLP). The primary objective was to extract structured information from unstructured text such as invoices, receipts, and business documents.

By combining regular expressions with spaCy's Named Entity Recognition (NER), the notebook transforms raw textual data into machine-readable structured outputs suitable for automation, analytics, and downstream business applications.

---

## Objectives

* Extract key information from textual documents.
* Identify and normalize dates, monetary values, and invoice numbers.
* Detect named entities such as people, organizations, locations, dates, and monetary amounts.
* Visualize recognized entities for analysis and debugging.
* Build a reusable document-processing pipeline.
* Export extracted results in JSON format for integration with other systems.

---

## Technologies and Libraries Used

* **Python** – Core programming language.
* **re (Regular Expressions)** – Pattern-based text extraction.
* **spaCy** – Industrial-strength NLP library.
* **displaCy** – Visualization tool for named entities.
* **JSON** – Structured data storage and exchange.

---

## Major Components Implemented

### 1. Date Extraction

A custom function was developed to identify dates in multiple formats, including:

* `MM/DD/YYYY`
* `DD-MM-YYYY`
* `Month DD, YYYY`
* `YYYY-MM-DD` (ISO format)

This enables robust handling of dates commonly found in invoices and business documents.

---

### 2. Monetary Amount Extraction

A regex-based parser was implemented to extract currency values from text, supporting formats such as:

* `$1,250.50`
* `$1250`
* `112.45`

Extracted values were cleaned and converted into floating-point numbers for numerical processing.

---

### 3. Invoice Number Extraction

A flexible invoice identifier extractor was built to recognize multiple invoice formats, including:

* `INV-2024-001`
* `#12345`
* `ORDER-ABC123`
* `Invoice Number: XYZ123`

This ensures compatibility across different invoice templates and naming conventions.

---

### 4. Named Entity Recognition (NER)

Using spaCy, the notebook extracts important business entities from text, including:

* **Persons**
* **Organizations**
* **Locations**
* **Dates**
* **Monetary Values**

This converts unstructured text into structured semantic information.

---

### 5. Entity Visualization

Named entities were visually highlighted using spaCy's `displacy` renderer. This provided:

* Interactive in-notebook visualization
* HTML export for external viewing and reporting
* Improved interpretability and debugging of NER results

---

### 6. Document Processing Pipeline

A consolidated processing function was created to:

* Accept raw document text
* Apply all extraction modules
* Aggregate results into a structured dictionary
* Standardize output format

This modular design improves reusability and scalability.

---

### 7. JSON Export

The final extracted information was serialized into JSON format, enabling:

* Easy storage
* API integration
* Database ingestion
* Interoperability with external systems

---

## Sample Information Extracted

The pipeline can successfully identify:

* Invoice numbers
* Dates
* Monetary amounts
* Customer or vendor names
* Organizations
* Geographic locations

---

## Key Learning Outcomes

* Practical use of regular expressions for pattern matching.
* Application of spaCy for industrial NLP tasks.
* Understanding of Named Entity Recognition (NER).
* Techniques for extracting structured data from unstructured text.
* Building modular and reusable NLP pipelines.
* Exporting processed data in JSON format.
* Visualizing NLP outputs for validation and presentation.

---

## Real-World Applications

* Invoice automation
* Receipt processing
* Financial document analysis
* Intelligent document processing (IDP)
* Business process automation
* Data extraction for ERP systems
* OCR post-processing pipelines

---

## Conclusion

This notebook successfully demonstrates the design and implementation of an end-to-end NLP-based document information extraction system. By integrating rule-based methods with machine learning-driven entity recognition, the solution achieves both flexibility and accuracy.

The resulting pipeline provides a strong foundation for real-world intelligent document processing applications, particularly in finance, accounting, and enterprise automation domains.

It can be further extended with OCR integration, document classification, validation rules, and deployment as an API or production-ready application.
